# Topic: SQL: User Growth Over Time & MoM Calculations

## Definition
*   **User Growth Analysis:** Measuring the rate at which a user base expands or contracts over standard time periods (days, weeks, months).
*   **Month-over-Month (MoM):** A percentage metric comparing the current month's new users (or total users) to the previous month's.

## Why Interviewers Ask This
*   **Core Product Metric:** Growth is the primary KPI for almost all consumer tech and SaaS products.
*   **SQL Proficiency:** Tests your ability to combine Common Table Expressions (CTEs), aggregate functions, and window functions seamlessly.
*   **Edge Case Awareness:** Reveals if you anticipate common data traps like division-by-zero or date granularity issues.

## Core Concepts
*   **Date Truncation:** Grouping exact timestamps into consistent analytical buckets (e.g., start of the month/week).
*   **Window Functions (LAG):** Accessing data from a previous row in the same result set without self-joining.
*   **Window Functions (SUM OVER):** Calculating running totals (cumulative sums) over an ordered partition.

## When to Use
*   Calculating new signups per day, week, or month.
*   Computing Week-over-Week (WoW) or Month-over-Month (MoM) growth rates.
*   Visualizing cumulative user counts over time to show overall platform size.

## Advantages
*   **Standardized Benchmarking:** MoM and WoW allow for apples-to-apples comparisons across different periods.
*   **Trend Identification:** Smooths out daily volatility to reveal broader product adoption trends.
*   **Efficiency:** Using `LAG()` is computationally much faster and more readable than writing complex self-joins to get previous period data.

## Limitations
*   **Ignores Churn:** Measuring "new users" doesn't account for users leaving; high growth can mask high churn.
*   **Seasonality Masking:** Simple MoM might look bad in typically slow months; Year-over-Year (YoY) is sometimes needed for context.
*   **Vanity Metric Risk:** Cumulative users will always go up; active users (MAU/DAU) are usually a better health indicator.

## Common Comparisons
*   **LAG() vs. Self-Join:** `LAG()` is cleaner and faster for sequential row comparison; Self-joins are better for complex, non-sequential relational lookups.
*   **MoM Growth vs. Absolute Growth:** MoM (e.g., +10%) shows momentum; absolute (e.g., +1,000 users) shows raw scale. Both are needed for context.
*   **New Users vs. Total Base Growth:** Don't confuse the growth rate of *new signups* with the growth rate of the *entire user base*.

## Common Interview Traps
*   **Division by Zero:** Failing to use `NULLIF(prev_val, 0)` when the prior period had zero users.
*   **Incorrect Date Granularity:** Forgetting to use `DATE_TRUNC` and accidentally grouping by daily timestamps.
*   **Counting Total Instead of New:** Not filtering by `signup_date` when the prompt specifically asks for *new* users.
*   **Ignoring Negative Growth:** Forgetting that the formula naturally yields a negative percentage if signups decline.

## SQL Syntax & Important Formula
*   **Date Truncation:** `DATE_TRUNC('month', signup_date)`
*   **Accessing Previous Row:** `LAG(new_users) OVER (ORDER BY signup_month)`
*   **Cumulative Total:** `SUM(new_users) OVER (ORDER BY signup_month)`
*   **MoM Formula:** `(current - previous) / previous * 100`
    *   *Safe SQL Implementation:* `100.0 * (curr - prev) / NULLIF(prev, 0)`

## 45-Second Interview Answer
"To calculate MoM user growth, I use a two-step CTE approach. First, I aggregate the data by truncating the signup dates to the month level and counting new users. In the second step, I use the `LAG()` window function ordered by the signup month to pull the previous month's total directly next to the current month's total. Finally, I calculate the percentage change using `(current - previous) / previous`, wrapping the denominator in a `NULLIF` to prevent division-by-zero errors."

## Example Questions:

### Q1. Show the number of new paid users per week for Q1 2025.

**Ideal Answer:**
```sql
SELECT 
    DATE_ADD(signup_date, INTERVAL -WEEKDAY(signup_date) DAY) AS signup_week,
    COUNT(user_id) AS new_paid_users
FROM users
WHERE plan = 'paid'
  AND signup_date >= '2025-01-01' 
  AND signup_date < '2025-04-01'
GROUP BY signup_week
ORDER BY signup_week;
```
*   **Common Mistakes:** Forgetting to filter by `plan = 'paid'`, or using `BETWEEN` for dates which can accidentally include midnight of April 1st.
*   **Interviewer Follow-up:** "How would you modify this to show weeks with zero signups?" (Answer: Generate a recursive CTE date series and LEFT JOIN the users table to it).

### Q2. Calculate the week-over-week growth rate of new signups.

**Ideal Answer:**
```sql
WITH weekly_signups AS (
    SELECT 
        DATE_ADD(signup_date, INTERVAL -WEEKDAY(signup_date) DAY) AS signup_week,
        COUNT(user_id) AS new_users
    FROM users
    GROUP BY signup_week
),
with_lag AS (
    SELECT 
        signup_week,
        new_users,
        LAG(new_users) OVER (ORDER BY signup_week) AS prev_week_users
    FROM weekly_signups
)
SELECT 
    signup_week,
    new_users,
    prev_week_users,
    ROUND(100.0 * (new_users - prev_week_users) / NULLIF(prev_week_users, 0), 1) AS wow_growth_pct
FROM with_lag
ORDER BY signup_week;
```
*   **Common Mistakes:** Missing the `NULLIF` in the denominator, or forgetting to order the window function by time.
*   **Interviewer Follow-up:** "What happens to the first week in the dataset?" (Answer: `prev_week_users` evaluates to NULL, so the growth rate evaluates to NULL, which is expected).

### Q3. Find the month with the highest month-over-month growth rate.

**Ideal Answer:**
```sql
WITH monthly_signups AS (
    SELECT 
        DATE_FORMAT(signup_date, '%Y-%m-01') AS signup_month, 
        COUNT(user_id) AS new_users
    FROM users 
    GROUP BY signup_month
),
with_lag AS (
    SELECT 
        signup_month, 
        new_users,
        LAG(new_users) OVER (ORDER BY signup_month) AS prev_users
    FROM monthly_signups
),
growth_calc AS (
    SELECT 
        signup_month,
        (100.0 * (new_users - prev_users) / NULLIF(prev_users, 0)) AS mom_growth
    FROM with_lag
)
SELECT 
    signup_month, 
    mom_growth
FROM growth_calc
WHERE mom_growth IS NOT NULL
ORDER BY mom_growth DESC
LIMIT 1;
```
*   **Common Mistakes:** Not filtering out the NULL growth rate from the very first month, which can cause unexpected sorting behaviors depending on the SQL dialect. 
*   **Interviewer Follow-up:** "If absolute user numbers are very small (e.g., jumping from 1 to 5 users), the MoM growth is 400%. How would you filter out 'noisy' months with low sample sizes?"

### Q4. Show cumulative user counts broken down by plan type (free vs paid) over time.

**Ideal Answer:**
```sql
WITH monthly_signups AS (
    SELECT 
        DATE_FORMAT(signup_date, '%Y-%m-01') AS signup_month,
        plan,
        COUNT(user_id) AS new_users
    FROM users
    GROUP BY signup_month, plan
)
SELECT 
    signup_month,
    plan,
    new_users,
    SUM(new_users) OVER (
        PARTITION BY plan 
        ORDER BY signup_month
    ) AS cumulative_users
FROM monthly_signups
ORDER BY plan, signup_month;
```
*   **Common Mistakes:** Forgetting the `PARTITION BY plan` inside the `SUM() OVER()` clause, which would result in a combined cumulative total instead of separate running totals for free and paid.
*   **Interviewer Follow-up:** "If no users sign up for a 'paid' plan in a specific month, how does this query behave?" (Answer: The month drops out for the 'paid' plan entirely. To fix it, you'd need a cross join of all months and all plans).